# Module 00 Lab — Build a Bare-Metal Agent

**Course:** AI Agents Teaching Platform  
**Module:** 00 — Mental Models  
**Estimated time:** 2–3 hours

---

### What you'll build

A working agent loop **from scratch** — no LangChain, no agent frameworks, no SDK abstractions beyond the API client. Just Python.

By the end you'll have:
- `tools.py` — a safe calculator tool
- `agent.py` — a ~50-line agent loop that calls the LLM, executes tools, and terminates correctly

### Constraints
- No LangChain, LangGraph, AutoGen, or similar frameworks
- Raw API calls only
- Loop must have a working termination condition **and** a `MAX_STEPS` guardrail

---

> **VS Code / local?** You can run this lab locally too — see the lab page on the platform for instructions.

## Step 1 — Setup

Run this cell once to install dependencies.

In [ ]:
%pip install -q anthropic

## Step 2 — API key

Paste your Anthropic API key below. It stays in this session only and is never sent anywhere except the Anthropic API.

Get a key at [console.anthropic.com](https://console.anthropic.com).

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("Key set ✓")

## Step 3 — Build `tools.py`

The agent needs at least one tool. We'll build a **safe calculator** — safe because it uses Python's AST parser instead of `eval()`. This matters: the expression comes from the model, which means it's untrusted input.

### Why not `eval(expression)`?

```python
eval("__import__('os').system('rm -rf /')")
```

`eval` would execute that. The AST approach only allows numbers and the four arithmetic operators — anything else returns an error string.

### TODO

The `_safe_eval` function is complete. Your job is to fill in `calculator()` so it:
1. Parses the expression string into an AST
2. Calls `_safe_eval` on the root node
3. Returns the result as a string
4. Catches any exception and returns a descriptive error string (never crashes)

In [ ]:
import ast
import operator as op

# Allowed operators — nothing else will execute
_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
}

def _safe_eval(node):
    """Recursively evaluate an AST node. Only numbers and +/-/* / are allowed."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


def calculator(args: dict) -> str:
    """Safe calculator tool. args must contain 'expression' key."""
    # TODO: implement the calculator
    # Hint: use ast.parse(expr, mode="eval").body to get the root node
    pass


# Tool registry — maps tool name to function
TOOLS = {"calculator": calculator}


# --- Quick self-test (run this cell to check your work) ---
assert calculator({"expression": "2 + 2"}) == "4", "Basic addition failed"
assert calculator({"expression": "10 / 2"}) == "5.0", "Division failed"
assert "error" in calculator({"expression": "__import__('os')"}).lower(), "Should return error for unsafe input"
print("calculator() tests passed ✓")

<details>
<summary>💡 Stuck? Reveal the calculator solution</summary>

```python
def calculator(args: dict) -> str:
    try:
        tree = ast.parse(args["expression"], mode="eval")
        result = _safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"calculator error: {e}"
```

</details>

## Step 4 — Build `agent.py`

Now the main loop. The agent:
1. Takes a goal from the user
2. Sends it (plus conversation history) to the LLM
3. The LLM replies with JSON: either a tool call or a final answer
4. If a tool call, execute the tool and append the result to the conversation
5. Repeat until the LLM says it's done, or `MAX_STEPS` is reached

### The protocol

We tell the model to reply in one of two JSON shapes:

**Tool call:**
```json
{"thought": "I need to compute 24 * 7", "action": "calculator", "args": {"expression": "24 * 7"}}
```

**Final answer:**
```json
{"thought": "I have the answer", "action": "final", "answer": "168"}
```

### The five TODOs

Fill in the five `pass` statements below.

In [ ]:
import json
import logging
import anthropic

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment
MODEL = "claude-haiku-4-5-20251001"  # fast and cheap for the lab
MAX_STEPS = 5

SYSTEM_PROMPT = """You are a small tool-using agent. Always reply with valid JSON — nothing else.

To use a tool:
  {"thought": "<your reasoning>", "action": "calculator", "args": {"expression": "<math expression>"}}

To give a final answer:
  {"thought": "<your reasoning>", "action": "final", "answer": "<your answer>"}
"""


def build_initial_messages(goal: str) -> list[dict]:
    """TODO 1: Return the starting messages list with the system prompt and user goal.
    
    The Anthropic API takes messages as a list of {role, content} dicts.
    The system prompt goes in a separate 'system' parameter (handled in call_model).
    """
    # TODO 1
    pass


def call_model(messages: list[dict]) -> str:
    """TODO 2: Call the Anthropic API and return the model's text response.
    
    Use client.messages.create(). Key parameters:
      model=MODEL, max_tokens=256, system=SYSTEM_PROMPT, messages=messages
    Return the text of the first content block.
    """
    # TODO 2
    pass


def parse_action(text: str) -> dict:
    """TODO 3: Parse the model's JSON response into a dict.
    
    If the text is not valid JSON, or has no 'action' key,
    return a safe default: {"action": "final", "answer": "<error message>"}
    so the loop always terminates cleanly.
    """
    # TODO 3
    pass


def run_agent(goal: str) -> str:
    """Main agent loop."""
    messages = build_initial_messages(goal)

    for step in range(MAX_STEPS):
        raw = call_model(messages)
        action = parse_action(raw)
        log.info("step %d | action: %s", step + 1, action.get("action"))

        # TODO 4: If action is 'final', return the answer
        # TODO 5: Otherwise, look up the tool in TOOLS, call it with action["args"],
        #         append the assistant message and the observation to messages, then continue
        pass

    return "max steps reached — try a simpler goal"

<details>
<summary>💡 Stuck? Reveal the full agent solution</summary>

```python
def build_initial_messages(goal: str) -> list[dict]:
    return [{"role": "user", "content": goal}]

def call_model(messages: list[dict]) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system=SYSTEM_PROMPT,
        messages=messages,
    )
    return response.content[0].text

def parse_action(text: str) -> dict:
    try:
        action = json.loads(text)
    except json.JSONDecodeError:
        return {"action": "final", "answer": "Model returned invalid JSON — stopping safely."}
    if "action" not in action:
        return {"action": "final", "answer": "No action key in model output — stopping safely."}
    return action

def run_agent(goal: str) -> str:
    messages = build_initial_messages(goal)
    for step in range(MAX_STEPS):
        raw = call_model(messages)
        action = parse_action(raw)
        log.info("step %d | action: %s", step + 1, action.get("action"))

        # TODO 4
        if action["action"] == "final":
            return action["answer"]

        # TODO 5
        tool_name = action["action"]
        if tool_name not in TOOLS:
            observation = f"Unknown tool: {tool_name}"
        else:
            observation = TOOLS[tool_name](action.get("args", {}))

        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "max steps reached — try a simpler goal"
```

</details>

## Step 5 — Test your agent

Once all TODOs are filled in, run the cells below to test your agent.

In [ ]:
# Test 1: basic arithmetic — the agent should use the calculator tool
result = run_agent("What is 24 multiplied by 7?")
print("Result:", result)
assert "168" in result, f"Expected 168 in result, got: {result}"
print("Test 1 passed ✓")

In [ ]:
# Test 2: multi-step — needs two tool calls
result = run_agent("What is (15 + 27) multiplied by 4?")
print("Result:", result)
assert "168" in result, f"Expected 168 in result, got: {result}"
print("Test 2 passed ✓")

In [ ]:
# Test 3: unknown tool — loop should not crash
result = run_agent("Search the web for the capital of France.")
print("Result:", result)  # should return something graceful, not raise an exception
print("Test 3 passed ✓ (didn't crash)")

## Step 6 — Experiments

These are optional but highly recommended — they'll make the concepts concrete.

### Experiment A: break the termination

Change `MAX_STEPS = 5` to `MAX_STEPS = 1` and run a multi-step goal. What happens?

### Experiment B: remove the JSON guard

In `parse_action`, remove the `try/except` block and replace the return with just `return json.loads(text)`. Then run a goal where the model is likely to stray from JSON (e.g. a conversational question). What error do you get? What does this tell you about why the guard exists?

### Experiment C: temperature

The Anthropic API accepts a `temperature` parameter in `client.messages.create()`. Add `temperature=1.0` and run the same arithmetic goal 5 times. Then try `temperature=0.0`. Do you see different behaviour? Why?

In [ ]:
# Scratch cell — use this for experiments


## Step 7 — Reflection (write your answers below)

These are the same questions from the `README.md` deliverable. Writing them here counts — copy into a README when you're done.

1. **What does your agent do?** (One sentence describing what goal it can handle and how.)

2. **What breaks if you remove the loop?** (What would happen if you just called `call_model` once with no loop?)

3. **What changes at `temperature=1.0` vs `0.0`?** (Based on your experiment above.)

*(Write your answers here — double-click to edit)*

1. 
2. 
3. 

## Done!

Head back to the platform and take the **Module Quiz** to mark Module 00 complete.

**Rubric reminder** — you need "Meets (3)" on all five criteria:
- Loop runs and terminates correctly
- You can explain every line of code cold
- You can draw and explain the agent loop correctly
- You can name a problem where a plain LLM call beats an agent loop
- README covers what breaks without the loop and temperature effects